#  Spark RCA - Root Cause Classification (SparkML)

Multi-class classifier to predict failure root causes from Spark execution features.

| Label | Root Cause |
|-------|------------------|
| 0 | Baseline |
| 1 | OOM |
| 2 | Data Skew |
| 3 | Serialization Failure |
| 4 | Network Timeout |
| 5 | Disk Space Failure |
| 6 | Metadata Failure |

## 1. Setup & Configuration

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, count, when, round as spark_round
from pyspark.sql.types import StringType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import (
    RandomForestClassifier,
    GBTClassifier,
    LogisticRegression,
    DecisionTreeClassifier
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

print("✓ Imports loaded")

✓ Imports loaded


In [2]:
# Configuration — HDFS paths (reading directly from cluster)
FEATURES_PATH = "hdfs://namenode:8020/project/features"
MODEL_OUTPUT_PATH = "hdfs://namenode:8020/project/models/rca_rf_model"
RESULTS_OUTPUT_PATH = "hdfs://namenode:8020/project/predictions"

# 25 ML features extracted by FeatureExtractor (Scala preprocessing)
FEATURE_COLUMNS = [
    # Stage-Level Aggregates (8)
    "mean_task_duration",
    "std_task_duration",
    "max_task_duration",
    "total_memory_spilled",
    "total_disk_spilled",
    "total_shuffle_read",
    "total_shuffle_write",
    "total_gc_time",
    # Structural Features (4)
    "total_stages",
    "failed_stages",
    "max_stage_parallelism",
    "stage_depth_of_failure",
    # Ratio Features (4)
    "completed_stage_ratio",
    "failed_stage_ratio",
    "spill_per_stage",
    "gc_per_stage",
    # Derived Features (9)
    "skew_index",
    "duration_variance",
    "max_min_duration_ratio",
    "spill_ratio",
    "disk_spill_ratio",
    "peak_memory_ratio",
    "gc_time_ratio",
    "task_failure_rate",
    "retry_count"
]

LABEL_NAMES = {
    0: "Baseline",
    1: "OOM",
    2: "Data Skew",
    3: "Serialization Failure",
    4: "Network Timeout",
    5: "Disk Space Failure",
    6: "Metadata Failure"
}

print("✓ Configuration set")
print(f"  Features: {len(FEATURE_COLUMNS)} columns")
print(f"  Labels:   {len(LABEL_NAMES)} classes")

✓ Configuration set
  Features: 25 columns
  Labels:   7 classes


In [ ]:
# Create Spark Session with HDFS access
spark = SparkSession.builder \
    .appName("RCA-ML-Classification") \
    .master("yarn") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✓ SparkSession created: {spark.version}")
print(f"  Master: {spark.sparkContext.master}")
print(f"  HDFS: hdfs://namenode:8020")

✓ SparkSession created: 3.5.0
  Master: local[*]
  HDFS: hdfs://namenode:8020


## 2. Load & Explore Data

In [4]:
# Load features from HDFS
df = spark.read.parquet(FEATURES_PATH)
total_rows = df.count()
print(f"✓ Loaded {total_rows} rows from HDFS: {FEATURES_PATH}")
print(f"  Columns: {len(df.columns)}")
df.printSchema()

✓ Loaded 79 rows from HDFS: hdfs://namenode:8020/project/features
  Columns: 27
root
 |-- app_id: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- mean_task_duration: double (nullable = true)
 |-- std_task_duration: double (nullable = true)
 |-- max_task_duration: double (nullable = true)
 |-- total_memory_spilled: double (nullable = true)
 |-- total_disk_spilled: double (nullable = true)
 |-- total_shuffle_read: double (nullable = true)
 |-- total_shuffle_write: double (nullable = true)
 |-- total_gc_time: double (nullable = true)
 |-- total_stages: double (nullable = true)
 |-- failed_stages: double (nullable = true)
 |-- max_stage_parallelism: double (nullable = true)
 |-- stage_depth_of_failure: double (nullable = true)
 |-- completed_stage_ratio: double (nullable = true)
 |-- failed_stage_ratio: double (nullable = true)
 |-- spill_per_stage: double (nullable = true)
 |-- gc_per_stage: double (nullable = true)
 |-- skew_index: double (nullable = true)
 |-- durati

In [5]:
# Fill nulls and add label names
df = df.fillna(0, subset=FEATURE_COLUMNS)

label_name_udf = udf(lambda l: LABEL_NAMES.get(l, "Unknown"), StringType())
df = df.withColumn("label_name", label_name_udf(col("label")))

# Label distribution
print("Label Distribution:")
df.groupBy("label", "label_name").count().orderBy("label").show(truncate=False)

Label Distribution:
+-----+---------------------+-----+
|label|label_name           |count|
+-----+---------------------+-----+
|0    |Baseline             |12   |
|1    |OOM                  |10   |
|2    |Data Skew            |12   |
|3    |Serialization Failure|9    |
|4    |Network Timeout      |14   |
|5    |Disk Space Failure   |11   |
|6    |Metadata Failure     |11   |
+-----+---------------------+-----+



In [ ]:
# Feature statistics
print("Feature Statistics:")
df.select(FEATURE_COLUMNS).describe().show(truncate=False)

Feature Statistics:


## 3. Feature Engineering & Train/Test Split

In [ ]:
# Assemble features into vector
assembler = VectorAssembler(
    inputCols=FEATURE_COLUMNS,
    outputCol="raw_features",
    handleInvalid="keep"
)

# Scale features (standardization)
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

print("✓ VectorAssembler configured")
print("✓ StandardScaler configured")

In [ ]:
# 80/20 Train-Test Split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.cache()
test_df = test_df.cache()

train_count = train_df.count()
test_count = test_df.count()

print(f"Train set: {train_count} rows")
print(f"Test set:  {test_count} rows")

print("\nTrain label distribution:")
train_df.groupBy("label", "label_name").count().orderBy("label").show(truncate=False)

print("Test label distribution:")
test_df.groupBy("label", "label_name").count().orderBy("label").show(truncate=False)

### 3a. K-Fold Cross-Validation (RF) — Generalization Check

In [ ]:
# K-Fold Cross-Validation (k=5) using the FULL dataset
# This gives a more robust estimate of generalization than a single 80/20 split
print("Running 5-Fold Cross-Validation on Random Forest...")

cv_assembler = VectorAssembler(inputCols=FEATURE_COLUMNS, outputCol="raw_features", handleInvalid="keep")
cv_scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
cv_rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=10, seed=42)

cv_pipeline = Pipeline(stages=[cv_assembler, cv_scaler, cv_rf])

# ParamGrid (no tuning — just want cross-val score)
paramGrid = ParamGridBuilder().build()

evaluator_cv = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

crossval = CrossValidator(
    estimator=cv_pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_cv,
    numFolds=5,
    seed=42
)

cv_model = crossval.fit(df)  # fit on FULL dataset — CV handles internal splits

cv_f1_scores = cv_model.avgMetrics
print(f"\n5-Fold Cross-Validation Results (Random Forest):")
print(f"  Mean F1 Score:  {cv_f1_scores[0]:.4f}")
print(f"  → This is a better generalization estimate than a single test split")
print(f"  → If this is much lower than the test F1, the model is overfitting")

## 4. Model Training

### 4a. Random Forest Classifier

In [ ]:
# Helper function to evaluate models
def evaluate_model(predictions, model_name):
    evaluator_acc = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    )
    evaluator_f1 = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1"
    )
    evaluator_prec = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
    )
    evaluator_rec = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedRecall"
    )

    metrics = {
        "accuracy": evaluator_acc.evaluate(predictions),
        "f1": evaluator_f1.evaluate(predictions),
        "precision": evaluator_prec.evaluate(predictions),
        "recall": evaluator_rec.evaluate(predictions)
    }

    print(f"\n📊 {model_name} Results:")
    print(f"   Accuracy:  {metrics['accuracy']:.4f}")
    print(f"   F1 Score:  {metrics['f1']:.4f}")
    print(f"   Precision: {metrics['precision']:.4f}")
    print(f"   Recall:    {metrics['recall']:.4f}")

    return metrics

print("✓ Evaluation function defined")

In [ ]:
# Random Forest
print("Training Random Forest Classifier...")

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    maxBins=32,
    seed=42
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

rf_metrics = evaluate_model(rf_predictions, "Random Forest")

In [ ]:
# Feature Importance (Random Forest)
importances = rf_model.stages[-1].featureImportances.toArray()
feature_imp = sorted(
    zip(FEATURE_COLUMNS, importances),
    key=lambda x: x[1],
    reverse=True
)

print("\n🌲 Random Forest - Feature Importance:")
print(f"{'Feature':<30} {'Importance':>12}")
print("-" * 42)
for feat, imp in feature_imp:
    bar = '█' * int(imp * 40)
    print(f"{feat:<30} {imp:>10.4f}  {bar}")

In [ ]:
# Feature Importance Bar Chart
feat_names = [f[0] for f in feature_imp]
feat_vals = [f[1] for f in feature_imp]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(feat_names)), feat_vals, color='#2196F3')
ax.set_yticks(range(len(feat_names)))
ax.set_yticklabels(feat_names, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Random Forest - Feature Importance (25 Features)')
plt.tight_layout()
plt.show()
print("✓ Feature importance chart displayed")

### 4b. Decision Tree Classifier

In [ ]:
print("Training Decision Tree Classifier...")

dt = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    maxBins=32,
    impurity="entropy",
    seed=42
)

dt_pipeline = Pipeline(stages=[assembler, scaler, dt])
dt_model = dt_pipeline.fit(train_df)
dt_predictions = dt_model.transform(test_df)

dt_metrics = evaluate_model(dt_predictions, "Decision Tree")

### 4c. Logistic Regression (Multinomial)

In [ ]:
print("Training Logistic Regression (Multinomial)...")

lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=100,
    family="multinomial",
    elasticNetParam=0.0,
    regParam=0.01
)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

lr_metrics = evaluate_model(lr_predictions, "Logistic Regression")

## 5. Model Comparison

In [ ]:
# Compare all models
results = {
    "Random Forest": rf_metrics,
    "Decision Tree": dt_metrics,
    "Logistic Regression": lr_metrics
}

print(f"\n{'Model':<25} {'Accuracy':>10} {'F1-Score':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 70)

best_model_name = None
best_f1 = -1.0

for name, metrics in results.items():
    print(f"{name:<25} {metrics['accuracy']:>10.4f} {metrics['f1']:>10.4f} "
          f"{metrics['precision']:>10.4f} {metrics['recall']:>10.4f}")
    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_model_name = name

print(f"\n★ Best Model: {best_model_name} (F1 = {best_f1:.4f})")

In [ ]:
# Model Comparison Bar Chart
model_names = list(results.keys())
metrics_names = ['accuracy', 'f1', 'precision', 'recall']
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for i, metric in enumerate(metrics_names):
    values = [results[m][metric] for m in model_names]
    ax.bar(x + i * width, values, width, label=metric.capitalize(), color=colors[i])

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()
print("✓ Model comparison chart displayed")

## 6. Confusion Matrix (Best Model)

In [ ]:
# Confusion Matrix
best_predictions = rf_predictions  # Use RF as default best

cm_df = best_predictions.groupBy("label") \
    .pivot("prediction") \
    .count() \
    .fillna(0) \
    .orderBy("label")

print("Confusion Matrix (rows=actual, cols=predicted):")
cm_df.show(truncate=False)

In [ ]:
# Per-Class Accuracy
print("Per-Class Accuracy:")
total_per_class = best_predictions.groupBy("label").count().collect()
correct_per_class = best_predictions.filter(col("prediction") == col("label")) \
    .groupBy("label").count().collect()

correct_dict = {row['label']: row['count'] for row in correct_per_class}

for row in sorted(total_per_class, key=lambda r: r['label']):
    label = row['label']
    total = row['count']
    correct = correct_dict.get(label, 0)
    acc = correct / total if total > 0 else 0
    name = LABEL_NAMES.get(int(label), 'Unknown')
    print(f"  Label {int(label)} ({name:<20}): {correct}/{total} = {acc:.2%}")

In [ ]:
# Confusion Matrix Heatmap
cm_pandas = cm_df.toPandas()
cm_pandas = cm_pandas.set_index('label')
# Ensure all 7 labels appear as columns (fill missing with 0)
for lbl in range(7):
    col_name = str(float(lbl))
    if col_name not in cm_pandas.columns:
        cm_pandas[col_name] = 0
cm_pandas = cm_pandas.reindex(sorted(cm_pandas.columns, key=float), axis=1)
cm_array = cm_pandas.values.astype(float)

label_display = [f"{i} ({LABEL_NAMES[i][:8]})" for i in range(len(cm_pandas.index))]
pred_display = [str(int(float(c))) for c in cm_pandas.columns]

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(cm_array, cmap='Blues')

ax.set_xticks(range(len(pred_display)))
ax.set_yticks(range(len(label_display)))
ax.set_xticklabels(pred_display, fontsize=9)
ax.set_yticklabels(label_display, fontsize=9)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
ax.set_title('Confusion Matrix - Random Forest')

for i in range(cm_array.shape[0]):
    for j in range(cm_array.shape[1]):
        val = int(cm_array[i, j])
        color = 'white' if val > cm_array.max() / 2 else 'black'
        ax.text(j, i, val, ha='center', va='center', color=color, fontsize=11)

plt.colorbar(im)
plt.tight_layout()
plt.show()
print("✓ Confusion matrix heatmap displayed")

## 7. Ground Truth Table

In [ ]:
# Ground Truth vs Predictions
ground_truth = best_predictions.select(
    "app_id", "label",
    col("prediction").cast("int").alias("predicted_label")
)

ground_truth = ground_truth.withColumn(
    "label_name", label_name_udf(col("label"))
).withColumn(
    "predicted_name", label_name_udf(col("predicted_label"))
).withColumn(
    "correct",
    when(col("label") == col("predicted_label"), "✓").otherwise("✗")
).orderBy("label", "app_id")

print("Ground Truth vs Predictions:")
ground_truth.show(100, truncate=False)

## 8. Save Model & Predictions

In [ ]:
# Save best model to HDFS
rf_model.write().overwrite().save(MODEL_OUTPUT_PATH)
print(f"✓ Model saved to HDFS: {MODEL_OUTPUT_PATH}")

# Save predictions to HDFS
best_predictions.select(
    "app_id", "label", "prediction", "features", "probability"
).write.mode("overwrite").parquet(RESULTS_OUTPUT_PATH)
print(f"✓ Predictions saved to HDFS: {RESULTS_OUTPUT_PATH}")

## 9. Holdout Evaluation — Unseen Test Data

In [ ]:
# Load holdout features (
HOLDOUT_FEATURES_PATH = "hdfs://namenode:8020/project/holdout-features"

from pyspark.ml import PipelineModel

print("Loading unseen holdout test data from HDFS...")
holdout_df = spark.read.parquet(HOLDOUT_FEATURES_PATH)
holdout_df = holdout_df.fillna(0, subset=FEATURE_COLUMNS)
holdout_df = holdout_df.withColumn("label_name", label_name_udf(col("label")))
holdout_count = holdout_df.count()
print(f"✓ Loaded {holdout_count} holdout samples from: {HOLDOUT_FEATURES_PATH}")

print("\nHoldout Label Distribution:")
holdout_df.groupBy("label", "label_name").count().orderBy("label").show(truncate=False)

# Load saved RF model
print("Loading saved Random Forest model...")
saved_rf_model = PipelineModel.load(MODEL_OUTPUT_PATH)
print(f"✓ Model loaded from: {MODEL_OUTPUT_PATH}")

# Predict on holdout data
holdout_predictions = saved_rf_model.transform(holdout_df)
holdout_metrics = evaluate_model(holdout_predictions, "RF on UNSEEN Holdout Data")

# Compare with original test metrics
print(f"\n📊 Generalization Comparison:")
print(f"  {'Metric':<12} {'Original Test':>15} {'Holdout (Unseen)':>18} {'Diff':>8}")
print(f"  {'-'*56}")
for metric in ['accuracy', 'f1', 'precision', 'recall']:
    orig = rf_metrics[metric]
    hold = holdout_metrics[metric]
    diff = hold - orig
    print(f"  {metric:<12} {orig:>15.4f} {hold:>18.4f} {diff:>+8.4f}")

if holdout_metrics['f1'] >= rf_metrics['f1'] * 0.85:
    print("\n✅ Model GENERALIZES well — holdout F1 within 15% of original")
else:
    print("\n⚠️  Model may be OVERFITTING — holdout F1 dropped significantly")

In [ ]:
# Holdout Confusion Matrix
holdout_cm = holdout_predictions.groupBy("label").pivot("prediction").count().fillna(0).orderBy("label")
print("Holdout Confusion Matrix (rows=actual, cols=predicted):")
holdout_cm.show(truncate=False)

# Per-Class Holdout Accuracy
print("Holdout Per-Class Accuracy:")
h_total = holdout_predictions.groupBy("label").count().collect()
h_correct = holdout_predictions.filter(col("prediction") == col("label")).groupBy("label").count().collect()
h_correct_dict = {row['label']: row['count'] for row in h_correct}

for row in sorted(h_total, key=lambda r: r['label']):
    label = row['label']
    total = row['count']
    correct = h_correct_dict.get(label, 0)
    acc = correct / total if total > 0 else 0
    name = LABEL_NAMES.get(int(label), 'Unknown')
    print(f"  Label {int(label)} ({name:<20}): {correct}/{total} = {acc:.2%}")

# Holdout Ground Truth Table
holdout_gt = holdout_predictions.select(
    "app_id", "label",
    col("prediction").cast("int").alias("predicted_label")
).withColumn("label_name", label_name_udf(col("label"))
).withColumn("predicted_name", label_name_udf(col("predicted_label"))
).withColumn("correct", when(col("label") == col("predicted_label"), "✓").otherwise("✗")
).orderBy("label", "app_id")

print("\nHoldout Ground Truth vs Predictions:")
holdout_gt.show(100, truncate=False)

In [ ]:
print("=" * 70)
print("ML PIPELINE COMPLETE")
print("=" * 70)
print(f"  Total training samples:    {total_rows}")
print(f"  Training samples:          {train_count}")
print(f"  Test samples:              {test_count}")
print(f"  Holdout (unseen) samples:  {holdout_count}")
print(f"  5-Fold CV F1 (RF):         {cv_f1_scores[0]:.4f}")
print(f"  Best Model:                {best_model_name}")
print(f"  Original Test F1:          {best_f1:.4f}")
print(f"  Holdout F1:                {holdout_metrics['f1']:.4f}")
print(f"  Model saved to:            {MODEL_OUTPUT_PATH}")
print(f"  Predictions at:            {RESULTS_OUTPUT_PATH}")
print("=" * 70)

In [ ]:
# Stop Spark session
spark.stop()
print("✓ Spark session stopped")